In [3]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm

In [2]:
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Zebrafish


In [4]:
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Zebrafish/Zebrafish_phenotype_chemicalentity/Zebrafish_phenotype_chemicalentity.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'species'
]

In [5]:
#

In [6]:
# Zebrafish = pd.read_csv(
#     f'{BASE_DIR}data_collection/databases_for_mapping/ncbi/Danio_rerio.gene_info',
#     sep='\t',
    
# )
# # Zebrafish["LocusTag"] = Zebrafish["LocusTag"].str.replace("Dmel_", "", regex=False)

# # Zebrafish_symbol2Locus_dict = dict(zip(Zebrafish['Symbol'], Zebrafish['LocusTag']))
# # Zebrafish
# # Zebrafish_symbol2Locus_dict
# Zebrafish

# monarch

In [7]:
monarch = pd.read_csv(PROC_DIR + 'Monarch/Monarch_final/Zebrafish/Zebra_Phenotype_ChemicalEntity.csv')

monarch.columns = monarch.columns.str.lower()
monarch = monarch[~monarch['head'].isna()]
# monarch = monarch.rename(columns={'headtype': 'head_type', 'tailtype': 'tail_type'})

monarch['kg_type'] = 'Generalised'
monarch['kg_source'] = 'Monarch_KG'
monarch['species'] = 'D.rerio'
print(f"monarch: {len(monarch):,} rows")
monarch

monarch: 702 rows


,head,relation,tail,head_type,relation_type,tail_type,relation_source,head_detail_name,tail_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,tail_iupac_name,species,kg_type,kg_source
0,ZP:0020710,Phenotype_ChemicalEntity,951,Phenotype,related_to,ChemicalEntity,infores:upheno,"noradrenaline brain decreased amount, abnormal",noradrenaline,Danio rerio,NaN,ZFIN_ID,PubChem,"4-(2-amino-1-hydroxyethyl)benzene-1,2-diol",D.rerio,Generalised,Monarch_KG
1,ZP:0020711,Phenotype_ChemicalEntity,5202,Phenotype,related_to,ChemicalEntity,infores:upheno,"serotonin brain decreased amount, abnormal",serotonin,Danio rerio,NaN,ZFIN_ID,PubChem,3-(2-aminoethyl)-1H-indol-5-ol,D.rerio,Generalised,Monarch_KG
2,ZP:0020712,Phenotype_ChemicalEntity,1826,Phenotype,related_to,ChemicalEntity,infores:upheno,(5-hydroxyindol-3-yl)acetic acid brain increas...,(5-hydroxyindol-3-yl)acetic acid,Danio rerio,NaN,ZFIN_ID,PubChem,2-(5-hydroxy-1H-indol-3-yl)acetic acid,D.rerio,Generalised,Monarch_KG
3,ZP:0020713,Phenotype_ChemicalEntity,681,Phenotype,related_to,ChemicalEntity,infores:upheno,"dopamine brain decreased amount, abnormal",dopamine,Danio rerio,NaN,ZFIN_ID,PubChem,"4-(2-aminoethyl)benzene-1,2-diol",D.rerio,Generalised,Monarch_KG
4,ZP:0021626,Phenotype_ChemicalEntity,681,Phenotype,related_to,ChemicalEntity,infores:upheno,"dopamine whole organism amount, abnormal",dopamine,Danio rerio,NaN,ZFIN_ID,PubChem,"4-(2-aminoethyl)benzene-1,2-diol",D.rerio,Generalised,Monarch_KG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
697,ZP:0018886,Phenotype_ChemicalEntity,617,Phenotype,related_to,ChemicalEntity,infores:upheno,"serine liver decreased amount, abnormal",serine,Danio rerio,NaN,ZFIN_ID,PubChem,2-amino-3-hydroxypropanoic acid,D.rerio,Generalised,Monarch_KG
698,ZP:0018888,Phenotype_ChemicalEntity,876,Phenotype,related_to,ChemicalEntity,infores:upheno,"methionine liver decreased amount, abnormal",methionine,Danio rerio,NaN,ZFIN_ID,PubChem,2-amino-4-methylsulfanylbutanoic acid,D.rerio,Generalised,Monarch_KG
699,ZP:0019009,Phenotype_ChemicalEntity,5538,Phenotype,related_to,ChemicalEntity,infores:upheno,"retinoic acid whole organism increased amount,...",retinoic acid,Danio rerio,NaN,ZFIN_ID,PubChem,"3,7-dimethyl-9-(2,6,6-trimethylcyclohexen-1-yl...",D.rerio,Generalised,Monarch_KG
700,ZP:0019312,Phenotype_ChemicalEntity,24742074,Phenotype,related_to,ChemicalEntity,infores:upheno,"1-phosphatidyl-1D-myo-inositol 4,5-bisphosphat...","1-phosphatidyl-1D-myo-inositol 4,5-bisphosphate",Danio rerio,NaN,ZFIN_ID,PubChem,"[(1R,2S,3R,4R,5S,6R)-2,3,5-trihydroxy-4-[[2-[(...",D.rerio,Generalised,Monarch_KG


In [8]:
monarch[monarch['head_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,relation_source,head_detail_name,tail_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,tail_iupac_name,species,kg_type,kg_source


# Consolidate into Unified Schem

In [9]:
# List all source DataFrames to include
source_dfs = [
     monarch
    
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df

Consolidated rows: 702


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,ZP:0020710,Phenotype_ChemicalEntity,951,Phenotype,related_to,ChemicalEntity,Monarch_KG,Generalised,ZFIN_ID,PubChem,"noradrenaline brain decreased amount, abnormal",noradrenaline,D.rerio
1,ZP:0020711,Phenotype_ChemicalEntity,5202,Phenotype,related_to,ChemicalEntity,Monarch_KG,Generalised,ZFIN_ID,PubChem,"serotonin brain decreased amount, abnormal",serotonin,D.rerio
2,ZP:0020712,Phenotype_ChemicalEntity,1826,Phenotype,related_to,ChemicalEntity,Monarch_KG,Generalised,ZFIN_ID,PubChem,(5-hydroxyindol-3-yl)acetic acid brain increas...,(5-hydroxyindol-3-yl)acetic acid,D.rerio
3,ZP:0020713,Phenotype_ChemicalEntity,681,Phenotype,related_to,ChemicalEntity,Monarch_KG,Generalised,ZFIN_ID,PubChem,"dopamine brain decreased amount, abnormal",dopamine,D.rerio
4,ZP:0021626,Phenotype_ChemicalEntity,681,Phenotype,related_to,ChemicalEntity,Monarch_KG,Generalised,ZFIN_ID,PubChem,"dopamine whole organism amount, abnormal",dopamine,D.rerio
...,...,...,...,...,...,...,...,...,...,...,...,...,...
697,ZP:0018886,Phenotype_ChemicalEntity,617,Phenotype,related_to,ChemicalEntity,Monarch_KG,Generalised,ZFIN_ID,PubChem,"serine liver decreased amount, abnormal",serine,D.rerio
698,ZP:0018888,Phenotype_ChemicalEntity,876,Phenotype,related_to,ChemicalEntity,Monarch_KG,Generalised,ZFIN_ID,PubChem,"methionine liver decreased amount, abnormal",methionine,D.rerio
699,ZP:0019009,Phenotype_ChemicalEntity,5538,Phenotype,related_to,ChemicalEntity,Monarch_KG,Generalised,ZFIN_ID,PubChem,"retinoic acid whole organism increased amount,...",retinoic acid,D.rerio
700,ZP:0019312,Phenotype_ChemicalEntity,24742074,Phenotype,related_to,ChemicalEntity,Monarch_KG,Generalised,ZFIN_ID,PubChem,"1-phosphatidyl-1D-myo-inositol 4,5-bisphosphat...","1-phosphatidyl-1D-myo-inositol 4,5-bisphosphate",D.rerio


# Sanity Check — Distinct Values

In [10]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'Phenotype_ChemicalEntity'}
head_type           : {'Phenotype'}
tail_type           : {'ChemicalEntity'}
relation_type       : {'related_to'}
kg_source           : {'Monarch_KG'}
head_id_is          : {'ZFIN_ID'}
tail_id_is          : {'PubChem'}


In [11]:
# Step 4: drop unresolvable rows
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 0 unresolvable rows → 702 remaining


# NaN Audit (pre-dedup)

In [12]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


# Deduplication

In [19]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first',
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 702  |  After dedup: 702


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,ZP:0018464,Phenotype_ChemicalEntity,5997,Phenotype,related_to,ChemicalEntity,Monarch_KG,Generalised,ZFIN_ID,PubChem,"cholesterol whole organism increased amount, a...",cholesterol,D.rerio
1,ZP:0018873,Phenotype_ChemicalEntity,439155,Phenotype,related_to,ChemicalEntity,Monarch_KG,Generalised,ZFIN_ID,PubChem,S-adenosyl-L-homocysteine liver decreased amou...,S-adenosyl-L-homocysteine,D.rerio
2,ZP:0018874,Phenotype_ChemicalEntity,594,Phenotype,related_to,ChemicalEntity,Monarch_KG,Generalised,ZFIN_ID,PubChem,"cysteine liver decreased amount, abnormal",cysteine,D.rerio


In [20]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


In [21]:
print("kg_source values present:", set(final_df_dedup['kg_source']), final_df_dedup['kg_source'].value_counts())

kg_source values present: {'Monarch_KG'} kg_source
Monarch_KG    702
Name: count, dtype: int64


In [22]:
print("kg_source values present:", set(final_df_dedup['kg_type']), final_df_dedup['kg_type'].value_counts())

kg_source values present: {'Generalised'} kg_type
Generalised    702
Name: count, dtype: int64


In [23]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 702 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Zebrafish/Zebrafish_phenotype_chemicalentity/Zebrafish_phenotype_chemicalentity.csv


In [2]:
# OUT_PATH